In [3]:
import sys
sys.path.append("..")

import pandas as pd
from sklearn.linear_model import LogisticRegression

from src.data import load_student_data, create_target

df = load_student_data()
df = create_target(df)

print(df.shape)
df.head()

(649, 34)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3,needs_support
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,3,4,1,1,3,4,0,11,11,0
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,3,3,1,1,3,2,9,11,11,0
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,3,2,2,3,3,6,12,13,12,0
3,GP,F,15,U,GT3,T,4,2,health,services,...,2,2,1,1,5,0,14,14,14,0
4,GP,F,16,U,GT3,T,3,3,other,other,...,3,2,1,2,5,0,11,13,13,0


In [4]:
from src.preprocessing import build_preprocessor

print("Preprocessing file imported successfully!")

Preprocessing file imported successfully!


In [5]:
X_base = df.drop(columns=["G3", "needs_support"])

X_early = X_base.drop(columns=["G1", "G2"])
X_progress = X_base.copy()

y = df["needs_support"]

print("Base:", X_base.shape)
print("Early-warning:", X_early.shape)
print("Progress-informed:", X_progress.shape)

Base: (649, 32)
Early-warning: (649, 30)
Progress-informed: (649, 32)


In [7]:
numeric_features_early = X_early.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features_early = X_early.select_dtypes(include=["object", "string"]).columns.tolist()

print("Numeric:", len(numeric_features_early))
print("Categorical:", len(categorical_features_early))

print(categorical_features_early[:10])

Numeric: 13
Categorical: 17
['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup']


In [8]:
from sklearn.model_selection import train_test_split
from src.config import RANDOM_STATE, TEST_SIZE

X_train_early, X_test_early, y_train, y_test = train_test_split(
    X_early,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Training set:", X_train_early.shape)
print("Testing set:", X_test_early.shape)
print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

Training set: (519, 30)
Testing set: (130, 30)
Training labels: (519,)
Testing labels: (130,)


In [9]:
from src.preprocessing import build_preprocessor

preprocessor_early = build_preprocessor(
    numeric_features_early,
    categorical_features_early
)

print(preprocessor_early)

ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['age', 'Medu', 'Fedu', 'traveltime',
                                  'studytime', 'failures', 'famrel', 'freetime',
                                  'goout', 'Dalc', 'Walc', 'health',
                                  'absences']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                    

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

logistic_early_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor_early),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE
            )
        )
    ]
)

logistic_early_pipeline.fit(X_train_early, y_train)

print("Early-warning logistic regression trained successfully!")

Early-warning logistic regression trained successfully!


In [11]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Predictions
train_pred = logistic_early_pipeline.predict(X_train_early)
test_pred = logistic_early_pipeline.predict(X_test_early)

# Probabilities for ROC-AUC
test_prob = logistic_early_pipeline.predict_proba(X_test_early)[:, 1]

# Create result dictionary
logistic_early_result = {
    "experiment_name": "early_warning",
    "model_name": "logistic_regression",
    "training_accuracy": accuracy_score(y_train, train_pred),
    "testing_accuracy": accuracy_score(y_test, test_pred),
    "support_precision": precision_score(y_test, test_pred, pos_label=1),
    "support_recall": recall_score(y_test, test_pred, pos_label=1),
    "support_f1": f1_score(y_test, test_pred, pos_label=1),
    "roc_auc": roc_auc_score(y_test, test_prob),
    "confusion_matrix": confusion_matrix(y_test, test_pred)
}

logistic_early_result

{'experiment_name': 'early_warning',
 'model_name': 'logistic_regression',
 'training_accuracy': 0.8786127167630058,
 'testing_accuracy': 0.8461538461538461,
 'support_precision': 0.5,
 'support_recall': 0.15,
 'support_f1': 0.23076923076923078,
 'roc_auc': 0.7659090909090909,
 'confusion_matrix': array([[107,   3],
        [ 17,   3]])}

In [12]:
logistic_early_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor_early),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE,
                class_weight="balanced"
            )
        )
    ]
)

logistic_early_pipeline.fit(X_train_early, y_train)

print("Balanced early-warning logistic regression trained successfully!")

Balanced early-warning logistic regression trained successfully!


In [13]:
train_pred = logistic_early_pipeline.predict(X_train_early)
test_pred = logistic_early_pipeline.predict(X_test_early)
test_prob = logistic_early_pipeline.predict_proba(X_test_early)[:, 1]

logistic_early_result = {
    "experiment_name": "early_warning",
    "model_name": "logistic_regression",
    "training_accuracy": accuracy_score(y_train, train_pred),
    "testing_accuracy": accuracy_score(y_test, test_pred),
    "support_precision": precision_score(y_test, test_pred, pos_label=1),
    "support_recall": recall_score(y_test, test_pred, pos_label=1),
    "support_f1": f1_score(y_test, test_pred, pos_label=1),
    "roc_auc": roc_auc_score(y_test, test_prob),
    "confusion_matrix": confusion_matrix(y_test, test_pred)
}

logistic_early_result

{'experiment_name': 'early_warning',
 'model_name': 'logistic_regression',
 'training_accuracy': 0.8111753371868978,
 'testing_accuracy': 0.7692307692307693,
 'support_precision': 0.35294117647058826,
 'support_recall': 0.6,
 'support_f1': 0.4444444444444444,
 'roc_auc': 0.775909090909091,
 'confusion_matrix': array([[88, 22],
        [ 8, 12]])}

In [14]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from src.config import CV_FOLDS

cv = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)

cv_results = cross_validate(
    logistic_early_pipeline,
    X_early,
    y,
    cv=cv,
    scoring=["recall", "f1", "roc_auc"]
)

logistic_early_result["cv_mean"] = cv_results["test_recall"].mean()
logistic_early_result["cv_std"] = cv_results["test_recall"].std()

print("CV Recall Mean:", round(logistic_early_result["cv_mean"], 4))
print("CV Recall Std:", round(logistic_early_result["cv_std"], 4))

CV Recall Mean: 0.67
CV Recall Std: 0.1503


In [16]:
numeric_features_progress = X_progress.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features_progress = X_progress.select_dtypes(include=["object", "string"]).columns.tolist()

print("Numeric:", len(numeric_features_progress))
print("Categorical:", len(categorical_features_progress))

Numeric: 15
Categorical: 17


In [17]:
X_train_progress, X_test_progress, _, _ = train_test_split(
    X_progress,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

print(X_train_progress.shape)
print(X_test_progress.shape)

(519, 32)
(130, 32)


In [18]:
preprocessor_progress = build_preprocessor(
    numeric_features_progress,
    categorical_features_progress
)

logistic_progress_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor_progress),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE,
                class_weight="balanced"
            )
        )
    ]
)

logistic_progress_pipeline.fit(X_train_progress, y_train)

print("Progress-informed logistic regression trained successfully!")

Progress-informed logistic regression trained successfully!


In [19]:
train_pred_p = logistic_progress_pipeline.predict(X_train_progress)
test_pred_p = logistic_progress_pipeline.predict(X_test_progress)
test_prob_p = logistic_progress_pipeline.predict_proba(X_test_progress)[:, 1]

logistic_progress_result = {
    "experiment_name": "progress_informed",
    "model_name": "logistic_regression",
    "training_accuracy": accuracy_score(y_train, train_pred_p),
    "testing_accuracy": accuracy_score(y_test, test_pred_p),
    "support_precision": precision_score(y_test, test_pred_p, pos_label=1),
    "support_recall": recall_score(y_test, test_pred_p, pos_label=1),
    "support_f1": f1_score(y_test, test_pred_p, pos_label=1),
    "roc_auc": roc_auc_score(y_test, test_prob_p),
    "confusion_matrix": confusion_matrix(y_test, test_pred_p)
}

logistic_progress_result

{'experiment_name': 'progress_informed',
 'model_name': 'logistic_regression',
 'training_accuracy': 0.9421965317919075,
 'testing_accuracy': 0.9076923076923077,
 'support_precision': 0.6538461538461539,
 'support_recall': 0.85,
 'support_f1': 0.7391304347826086,
 'roc_auc': 0.9372727272727273,
 'confusion_matrix': array([[101,   9],
        [  3,  17]])}

In [20]:
results_df = pd.DataFrame([
    logistic_early_result,
    logistic_progress_result
])

results_df[[
    "experiment_name",
    "testing_accuracy",
    "support_precision",
    "support_recall",
    "support_f1",
    "roc_auc"
]]

,experiment_name,testing_accuracy,support_precision,support_recall,support_f1,roc_auc
0,early_warning,0.769231,0.352941,0.60,0.444444,0.775909
1,progress_informed,0.907692,0.653846,0.85,0.739130,0.937273


## Leakage Prevention

This workflow prevents data leakage by:

* creating the target `needs_support` from `G3`,
* removing `G3` from all model input features,
* removing `G1` and `G2` from the early-warning experiment,
* fitting preprocessing only on the training data,
* placing imputation, scaling, and one-hot encoding inside a scikit-learn `Pipeline`,
* and using the same shared stratified train-test split for all experiments.

This ensures that the logistic regression models do not have access to information derived from the target variable or from the testing data during training.


## Conclusion

Two logistic regression experiments were developed:

1. **Early-Warning Model** — uses only background and behavioral features and excludes `G1`, `G2`, and `G3`.
2. **Progress-Informed Model** — includes `G1` and `G2` while still excluding `G3`.

The balanced logistic regression approach was used to improve the detection of students who may require academic support. Model performance was evaluated using accuracy, precision, recall, F1-score, ROC-AUC, confusion matrices, and cross-validation recall scores.

These results provide a baseline that can be compared with the Decision Tree and Random Forest models developed by other team members.


In [21]:
results_df

,experiment_name,model_name,training_accuracy,testing_accuracy,support_precision,support_recall,support_f1,roc_auc,confusion_matrix,cv_mean,cv_std
0,early_warning,logistic_regression,0.811175,0.769231,0.352941,0.60,0.444444,0.775909,"[[88, 22], [8, 12]]",0.67,0.150333
1,progress_informed,logistic_regression,0.942197,0.907692,0.653846,0.85,0.739130,0.937273,"[[101, 9], [3, 17]]",NaN,NaN
